# 92 — Normalization (rank / percentile)

`91_distribution_analyses.ipynb` showed that several layers are heavily skewed or fat-tailed, so plain min-max scaling lets a handful of extreme cells silently dominate the composite score. This notebook rank-transforms every layer to a uniform `[0, 1]` percentile instead — same order, no long tails — and stacks all layers into a single `normalized.nc` in the processed folder that `93_scoring.ipynb` can consume.

In [ ]:
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
from scipy.stats import rankdata

from common import PROCESSED_DIR, load_layers, load_weights

## Load layers

Same selection as the scoring notebook — every non-zero-weighted entry in `weights.yaml` whose `.nc` file exists on disk.

In [ ]:
weights = load_weights()
layers = load_layers(weights)
print(f'loaded {len(layers)} of {len(weights)} layers')
list(layers)

## Percentile transformation

Every finite cell is ranked (ties averaged) and rescaled so the smallest cell maps to `0` and the largest to `1`. The result is uniform on `[0, 1]` by construction, which neutralises skew and outliers before they reach the weighted sum. NaN cells (ocean, country-coverage gaps) stay NaN, so per-cell weight renormalisation in `weighted_score` still works.

In [ ]:
def percentile_rank(da):
    """Rank-transform a DataArray to percentiles in ``[0, 1]``.

    Ties are broken by the average rank. NaN cells are preserved and do not
    participate in the ranking. The output shares the input's coords, dims
    and name so it can be dropped into any pipeline that already consumed
    the raw layer.
    """
    values = da.values.astype(float)
    flat = values.ravel()
    mask = np.isfinite(flat)
    out = np.full_like(flat, np.nan)
    n = int(mask.sum())
    if n > 1:
        out[mask] = (rankdata(flat[mask], method='average') - 1) / (n - 1)
    elif n == 1:
        out[mask] = 0.0
    return da.copy(data=out.reshape(values.shape))

normed = {name: percentile_rank(da) for name, da in layers.items()}

## Before / after

Left column: raw values on the layer's own units and range. Right column: percentile-transformed values on the shared `[0, 1]` axis. The right column should be flat — that is the transformation working as advertised.

In [ ]:
def _finite(da):
    """Return the DataArray's non-NaN values as a 1-D numpy array."""
    v = da.values.ravel()
    return v[np.isfinite(v)]

n = len(layers)
fig, axes = plt.subplots(n, 2, figsize=(10, 2.2 * n))
for i, name in enumerate(layers):
    axes[i, 0].hist(_finite(layers[name]), bins=60, color='steelblue', edgecolor='none')
    axes[i, 0].set_title(f'{name} — raw')
    axes[i, 0].set_yticks([])

    axes[i, 1].hist(_finite(normed[name]), bins=60, range=(0, 1), color='seagreen', edgecolor='none')
    axes[i, 1].set_title(f'{name} — percentile')
    axes[i, 1].set_xlim(0, 1)
    axes[i, 1].set_yticks([])
fig.tight_layout()

## Save

All percentile-transformed layers share the same `(lat, lon)` grid, so they combine cleanly into a single `xarray.Dataset` — one variable per layer. Written to `processed/normalized.nc` for downstream consumption.

In [ ]:
ds = xr.Dataset(normed)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
out = PROCESSED_DIR / 'normalized.nc'
if out.exists():
    out.unlink()
ds.to_netcdf(out)
print(f'wrote {out}')
ds